In [ ]:
import sys
import os
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings('ignore')

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)


In [2]:

# 2. Modular Architecture Imports
from src.data_engineering import transform_and_engineer_memory, handle_nans_and_split, create_purged_time_splits
from src.splines import fit_variance_baseline_and_derivatives
from src.models import train_damped_xgboost_spatial, generate_final_submission_spatial

print("Environment path set. Modules loaded successfully.")

Environment path set. Modules loaded successfully.


In [3]:
# Global Seed Lockdown for Manual Audit Reproducibility
RANDOM_SEED = 1
np.random.seed(RANDOM_SEED)

print("Master Seed Locked.")

Master Seed Locked.


In [ ]:
data_path = '../data/raw/dataset.csv'

# 1. Data Pipeline
print("Loading Data and Engineering Memory...")
df = pd.read_csv(data_path)
df_long = transform_and_engineer_memory(df)
train_df, inference_df = handle_nans_and_split(df_long)
cv_splits = create_purged_time_splits(train_df)

# 2. Mathematical Baseline
print("Fitting Mathematical Splines and Extracting Derivatives...")
train_df, inference_df = fit_variance_baseline_and_derivatives(train_df, inference_df)

# Drop any rows where the spline failed to fit due to extreme illiquidity (< 4 points)
train_df = train_df.dropna(subset=['w_baseline'])

# 3. ML Corrector
print("\nInitiating Spatial Damped XGBoost Engine...")
trained_xgb_models = train_damped_xgboost_spatial(train_df, cv_splits)

# 4. Final Inference Export
print("\nGenerating Final Damped Predictions...")
kaggle_sub = generate_final_submission_spatial(inference_df, trained_xgb_models)

# Sanity Check Output
print(f"\nFinal Submission Shape: {kaggle_sub.shape}")

# Save the submission to the root directory
kaggle_sub.to_csv('../submission.csv', index=False)
print("\nSubmission.csv generated successfully in the root directory. Pipeline execution complete.")

Loading Data and Engineering Memory...
Fitting Mathematical Splines and Extracting Derivatives...

Initiating Spatial Damped XGBoost Engine...
GLOBAL OOF DAMPED XGBoost IV RMSE: 0.0184517

Generating Final Damped Predictions...

Final Submission Shape: (5460, 2)

Submission.csv generated successfully in the root directory. Pipeline execution complete.
